# Análise exploratória de movimentações de reagentes — setembro de 2026

**Período:** 01/09/2026 a 30/09/2026  
**Objetivo:** explorar registros de movimentação, verificar qualidade básica e construir indicadores para controle de estoque.


## 1. Bibliotecas e carregamento

Nesta etapa, importamos `pandas` para manipulação dos dados e `matplotlib` para os gráficos. Os arquivos são lidos com separador `;`, conforme foram exportados.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PASTA = Path(".")
ARQ_MOV = PASTA / "movimentacoes_reagentes_setembro_2026.csv"
ARQ_ESTOQUE = PASTA / "estoque_inicial_reagentes_2026-09-01.csv"

mov = pd.read_csv(ARQ_MOV, sep=";", encoding="utf-8-sig")
estoque = pd.read_csv(ARQ_ESTOQUE, sep=";", encoding="utf-8-sig")

mov["data_hora"] = pd.to_datetime(mov["data_hora"], errors="coerce")
estoque["data_validade"] = pd.to_datetime(estoque["data_validade"], errors="coerce")
mov["quantidade"] = pd.to_numeric(mov["quantidade"], errors="coerce")
estoque["saldo_inicial"] = pd.to_numeric(estoque["saldo_inicial"], errors="coerce")

print("Movimentações:", mov.shape)
print("Lotes no estoque inicial:", estoque.shape)
display(mov.head())
display(estoque.head())

## 2. Validação dos dados

Antes de calcular indicadores, verificamos datas, IDs, valores ausentes, quantidades não positivas e se os lotes movimentados existem no arquivo de estoque inicial. Essas verificações ajudam a evitar conclusões baseadas em registros incompletos ou inconsistentes.

In [ ]:
TIPOS_VALIDOS = {"entrada", "retirada", "devolução", "descarte"}
inicio = pd.Timestamp("2026-09-01 00:00:00")
fim = pd.Timestamp("2026-09-30 23:59:59")

validacao = pd.Series({
    "Quantidade de movimentações": len(mov),
    "Quantidade de lotes iniciais": len(estoque),
    "Data mínima": mov["data_hora"].min(),
    "Data máxima": mov["data_hora"].max(),
    "IDs de movimentação duplicados": int(mov["movimentacao_id"].duplicated().sum()),
    "Células ausentes em movimentações": int(mov.isna().sum().sum()),
    "Células ausentes no estoque inicial": int(estoque.isna().sum().sum()),
    "Quantidades menores ou iguais a zero": int((mov["quantidade"] <= 0).sum()),
    "Tipos de movimentação desconhecidos": int((~mov["tipo_movimentacao"].isin(TIPOS_VALIDOS)).sum()),
    "Lotes sem correspondência no estoque inicial": int((~mov["lote_id"].isin(estoque["lote_id"])).sum()),
    "Registros fora de setembro de 2026": int((~mov["data_hora"].between(inicio, fim)).sum()),
})
display(validacao.to_frame("resultado"))

# Exibe os tipos de movimentação encontrados para inspeção.
display(mov["tipo_movimentacao"].value_counts(dropna=False).rename_axis("tipo").to_frame("operacoes"))

## 3. Preparação das variáveis

A variação de estoque é calculada com sinais: entradas e devoluções aumentam o saldo; retiradas e descartes reduzem. Essa regra representa a lógica adotada nesta análise e deve coincidir com as regras do sistema real antes de ser aplicada a dados operacionais.

In [ ]:
sinais = {"entrada": 1, "devolução": 1, "retirada": -1, "descarte": -1}
mov["variacao_estoque"] = mov["tipo_movimentacao"].map(sinais) * mov["quantidade"]
mov["data"] = mov["data_hora"].dt.date
mov["dia"] = mov["data_hora"].dt.floor("D")

# Verificação explícita de unidades: quantidades serão agregadas apenas dentro da mesma unidade.
display(mov[["data_hora", "tipo_movimentacao", "reagente", "lote_id", "quantidade", "unidade", "variacao_estoque"]].head())

## 4. Frequência dos tipos de movimentação

A contagem de operações permite comparar a frequência dos eventos. As quantidades são agregadas separadamente por unidade, porque litros, quilogramas e gramas não podem ser somados entre si.

In [ ]:
resumo_tipos = (
    mov.groupby("tipo_movimentacao")
       .agg(operacoes=("movimentacao_id", "count"))
       .sort_values("operacoes", ascending=False)
)
resumo_tipos["percentual_operacoes"] = 100 * resumo_tipos["operacoes"] / len(mov)
display(resumo_tipos.round(2))
ax = resumo_tipos["operacoes"].plot(kind="bar", title="Quantidade de operações por tipo", ylabel="Operações", xlabel="Tipo de movimentação", rot=30)
plt.tight_layout()
plt.show()

resumo_tipo_unidade = (
    mov.groupby(["tipo_movimentacao", "unidade"])
       .agg(operacoes=("movimentacao_id", "count"), quantidade=("quantidade", "sum"))
       .reset_index()
)
display(resumo_tipo_unidade.round(3))
resumo_tipos.to_csv("resultado_resumo_tipos.csv", encoding="utf-8-sig")
resumo_tipo_unidade.to_csv("resultado_tipo_unidade.csv", index=False, encoding="utf-8-sig")

## 5. Atividade diária

A série diária mostra como a frequência de registros variou durante o mês. Um pico de operações não explica, sozinho, o motivo da atividade; a interpretação deve considerar calendário experimental e contexto operacional.

In [ ]:
mov_por_dia = mov.groupby("dia").size().rename("operacoes").asfreq("D", fill_value=0)
display(mov_por_dia.to_frame().head())
ax = mov_por_dia.plot(kind="line", marker="o", title="Movimentações registradas por dia", ylabel="Operações", xlabel="Data", figsize=(11, 4))
plt.tight_layout()
plt.show()
print("Dia com mais operações:", mov_por_dia.idxmax().strftime("%d/%m/%Y"))
print("Número de operações nesse dia:", int(mov_por_dia.max()))
mov_por_dia.to_frame().to_csv("resultado_movimentacoes_diarias.csv", encoding="utf-8-sig")

## 6. Análise por reagente

Primeiro avaliamos a frequência de registros por reagente. Em seguida, analisamos as quantidades retiradas por reagente e unidade. Frequência de movimentações e quantidade retirada são indicadores diferentes: um material pode aparecer em muitas operações pequenas ou em poucas operações grandes.

In [ ]:
frequencia_reagente = (
    mov.groupby(["reagente", "unidade"])
       .agg(operacoes=("movimentacao_id", "count"))
       .sort_values("operacoes", ascending=False)
)
display(frequencia_reagente.head(10))
frequencia_reagente.head(10).sort_values("operacoes").plot(kind="barh", title="Reagentes com maior frequência de movimentação", ylabel="Reagente", xlabel="Operações", figsize=(9, 5))
plt.tight_layout()
plt.show()

retiradas = mov[mov["tipo_movimentacao"] == "retirada"]
retiradas_reagente = (
    retiradas.groupby(["reagente", "unidade"])
             .agg(operacoes_retirada=("movimentacao_id", "count"),
                  quantidade_retirada=("quantidade", "sum"))
             .sort_values("operacoes_retirada", ascending=False)
)
display(retiradas_reagente.round(3))
frequencia_reagente.to_csv("resultado_frequencia_reagente.csv", encoding="utf-8-sig")
retiradas_reagente.to_csv("resultado_retiradas_reagente.csv", encoding="utf-8-sig")

## 7. Retiradas por projeto

A contagem de retiradas por projeto ajuda a observar a distribuição das operações registradas. Quantidades devem ser mantidas separadas por unidade. Registros sem projeto associado devem ser contabilizados e investigados, em vez de atribuídos arbitrariamente.

In [ ]:
retiradas_projeto = (
    retiradas.assign(projeto=retiradas["projeto"].fillna("Não informado").replace("", "Não informado"))
             .groupby("projeto")
             .agg(operacoes=("movimentacao_id", "count"))
             .sort_values("operacoes", ascending=False)
)
display(retiradas_projeto)
retiradas_projeto.plot(kind="bar", title="Retiradas por projeto", ylabel="Operações", xlabel="Projeto", rot=35, figsize=(10, 4))
plt.tight_layout()
plt.show()

retiradas_projeto_unidade = (
    retiradas.assign(projeto=retiradas["projeto"].fillna("Não informado").replace("", "Não informado"))
             .groupby(["projeto", "unidade"])
             .agg(operacoes=("movimentacao_id", "count"), quantidade_retirada=("quantidade", "sum"))
             .reset_index()
)
display(retiradas_projeto_unidade.round(3))
retiradas_projeto.to_csv("resultado_retiradas_projeto.csv", encoding="utf-8-sig")
retiradas_projeto_unidade.to_csv("resultado_retiradas_projeto_unidade.csv", index=False, encoding="utf-8-sig")

## 8. Descartes

Os descartes são analisados por frequência, reagente, unidade e motivo. Esse indicador ajuda a organizar o acompanhamento, mas não permite concluir sozinho que houve desperdício evitável: é necessário conhecer a causa do descarte e o contexto do material.

In [ ]:
descartes = mov[mov["tipo_movimentacao"] == "descarte"].copy()
print("Operações de descarte:", len(descartes))
display(descartes.groupby("motivo").size().sort_values(ascending=False).rename("operacoes").to_frame())
display(descartes.groupby(["reagente", "unidade"]).agg(
    operacoes=("movimentacao_id", "count"),
    quantidade_descartada=("quantidade", "sum")
).sort_values("operacoes", ascending=False).round(3))
descartes.to_csv("resultado_registros_descarte.csv", index=False, encoding="utf-8-sig")

## 9. Saldos estimados por lote

O saldo final estimado é o saldo inicial acrescido das entradas e devoluções e reduzido pelas retiradas e descartes. O cálculo depende de o saldo inicial representar corretamente o começo do período e de todas as movimentações estarem registradas. Saldos negativos são sinalizadores de inconsistência a investigar, não devem ser corrigidos silenciosamente.

In [ ]:
variacao_lote = mov.groupby("lote_id")["variacao_estoque"].sum()
saldos = estoque[["reagente_id", "reagente", "lote_id", "unidade", "saldo_inicial", "data_validade"]].copy()
saldos["variacao_setembro"] = saldos["lote_id"].map(variacao_lote).fillna(0)
saldos["saldo_final_estimado"] = saldos["saldo_inicial"] + saldos["variacao_setembro"]
saldos["saldo_negativo"] = saldos["saldo_final_estimado"] < 0
display(saldos.sort_values("saldo_final_estimado").round(3))
print("Lotes com saldo final negativo:", int(saldos["saldo_negativo"].sum()))
display(saldos[saldos["saldo_negativo"]])
saldos.to_csv("resultado_saldos_estimados_lote.csv", index=False, encoding="utf-8-sig")

## 10. Vencimentos próximos

Listamos os lotes com validade entre outubro e novembro de 2026. É um indicador preventivo: proximidade do vencimento não significa necessariamente perda, pois a decisão depende de saldo, condições de armazenamento e demanda.

In [ ]:
vencimentos_proximos = saldos[
    saldos["data_validade"].between(pd.Timestamp("2026-10-01"), pd.Timestamp("2026-11-30"))
].sort_values("data_validade")
display(vencimentos_proximos[["reagente", "lote_id", "saldo_final_estimado", "unidade", "data_validade"]])
vencimentos_proximos.to_csv("resultado_vencimentos_proximos.csv", index=False, encoding="utf-8-sig")

## 11. Discussão e limitações

Interprete os resultados em conjunto: frequência mostra o volume de registros; quantidades precisam ser avaliadas por unidade; retiradas não equivalem necessariamente ao consumo efetivo; e saldos estimados dependem da completude do histórico e da validade do saldo inicial. Um único mês não é suficiente para estabelecer sazonalidade ou tendência de longo prazo.

Como os dados de entrada são sintéticos, os padrões e valores obtidos servem para demonstrar o método analítico, não para afirmar fatos sobre o laboratório. Para uso operacional ou em relatório como evidência real, substitua-os por dados autorizados e verificados, preservando a confidencialidade do laboratório.

## 12. Exportação das tabelas

As células anteriores exportam tabelas derivadas em CSV para que os resultados possam ser revisados e usados em etapas posteriores.

In [ ]:
print("Análise concluída. Os arquivos resultado_*.csv foram salvos na pasta de trabalho.")